# Controlled interpolation-scheme comparison for PCA / t-SNE / UMAP

NOFE initializes query-node features by inverse-distance-weighted (IDW) interpolation of the
*raw* input features over the `k_cross` nearest cross-graph neighbours (see
`ERA5_Graph_Generator.inverse_distance_interpolation` / `construct_output_node_features` in
`src/generator_utility.py`), then a trained graph network predicts the low-dim embedding at those
query nodes from the interpolated features plus local graph structure.

The existing baseline notebook (`rebuttal_distance_presevation_super_resolution.ipynb`) fits
PCA / t-SNE / UMAP on the source point cloud only, then carries the resulting embedding to query
locations with `scipy.interpolate.LinearNDInterpolator` (piecewise-linear interpolation over a
Delaunay triangulation) -- a different interpolation scheme than NOFE's own IDW.

This notebook adds a controlled comparison: for each of PCA / t-SNE / UMAP, the query-location
embedding is produced by **four** interpolation schemes applied to the same source embedding:

- `linear`  -- the existing `LinearNDInterpolator` baseline (kept for continuity)
- `idw`     -- the exact IDW scheme NOFE itself uses (same `k_cross` neighbours, same weights)
- `rbf`     -- thin-plate-spline RBF interpolation (`scipy.interpolate.RBFInterpolator`)
- `gp`      -- an ordinary-kriging stand-in: one independent Gaussian Process (RBF kernel + nugget)
  per output dimension (`sklearn.gaussian_process.GaussianProcessRegressor`)

For UMAP specifically, which (unlike PCA and t-SNE) has a native out-of-sample extension, we add a
fifth variant:

- `native` -- IDW-interpolate the *raw* input features to the query locations (the same
  interpolated point NOFE's model would see), then embed that point directly via UMAP's own
  fitted `.transform()` -- this mirrors NOFE's own pipeline shape (interpolate raw -> embed)
  most closely, rather than interpolating a precomputed embedding.

Note: PCA's transform is affine and IDW is a convex combination (weights sum to 1), so for PCA,
`idw` and `native`-style ("interpolate raw, then project") are mathematically identical -- this is
verified numerically below as a sanity check.


In [1]:
# torch, faiss (k-NN/perplexity graph construction) and numba-based internals (UMAP) all get
# loaded into one process here; with unconstrained thread pools that combination makes macOS
# abort with a duplicate-OpenMP-runtime error (kernel crash / segfault). Capping thread counts
# before any of those libraries import, plus the (unsafe but standard) escape hatch as a second
# line of defense, avoids it.
import os
os.environ.setdefault('KMP_DUPLICATE_LIB_OK', 'TRUE')
os.environ.setdefault('OMP_NUM_THREADS', '1')
os.environ.setdefault('MKL_NUM_THREADS', '1')
os.environ.setdefault('OPENBLAS_NUM_THREADS', '1')
os.environ.setdefault('NUMEXPR_NUM_THREADS', '1')
os.environ.setdefault('NUMBA_NUM_THREADS', '1')

import sys
import warnings
import numpy as np
import pandas as pd
import torch
from pathlib import Path

import subprocess
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import RBF as RBFKernel
from sklearn.exceptions import ConvergenceWarning
from scipy.interpolate import LinearNDInterpolator, RBFInterpolator
import umap
import phate
import anndata as ad
import concord as ccd
import synthia as syn

warnings.filterwarnings('ignore', category=ConvergenceWarning)
# GP kernel hyperparameter search occasionally produces harmless overflow in intermediate
# matmuls when the optimal length-scale collapses close to its lower bound; the resulting
# predictions are still finite and are additionally sanitized with np.nan_to_num below.
warnings.filterwarnings('ignore', category=RuntimeWarning, module='sklearn.*')

project_root = Path.cwd().parent.parent
sys.path.insert(0, str(project_root / 'src'))
sys.path.insert(0, str(project_root / 'ERA5' / 'scripts'))

from utilities import setup_device
from utilities_era5 import ERA5_Graph_Generator as Data_Generator

device = setup_device()
print(f'Using device: {device}')

using Device: mps
Using device: mps


In [2]:
# Trained NOFE weights for exactly this super-resolution config (confirmed by matching
# state-dict shapes: width=64, ker_in=7, known/unknown_node_dim=11, out_width=3 -- see
# ERA5/configurations/subgraph/model_configs/base2_d2.json and
# ERA5/configurations/subgraph/dataset_configs/sub_fixcrop_median_woPos_tkernel.json).
config = {
    'lat_min': 30,
    'lat_max': 70,
    'lon_min': -10,
    'lon_max': 30,
    'k_in': 5,
    'k_out': 5,
    'k_cross': 5,
    'perplexity': 20,
    'input_sampling_method': 'random',
    'output_sampling_method': 'random',
    'feature_mode': 'channel_only',
    'interpolate_graph_features': True,
    'vars': ['w', 'v', 'z', 'q', 'r', 't', 'd', 'o3', 'pv', 'u', 'vo'],
    'width': 64,
    'ker_width': 64,
    'depth': 2,
    'input_node_features': 11,
    'output_node_features': 11,
    'edge_attr': 7,
    'out_width': 3,
}

lat_min, lat_max = config['lat_min'], config['lat_max']
lon_min, lon_max = config['lon_min'], config['lon_max']
k_in, k_out, k_cross = config['k_in'], config['k_out'], config['k_cross']
perplexity = config['perplexity']
input_sampling_method = config['input_sampling_method']
output_sampling_method = config['output_sampling_method']
feature_mode = config['feature_mode']
interpolate_graph_features = config['interpolate_graph_features']
vars = config['vars']

from utilities import KernelNN_subgraph

nofe_model = KernelNN_subgraph(
    width=config['width'],
    ker_width=config['ker_width'],
    depth=config['depth'],
    ker_in=config['edge_attr'],
    known_node_dim=config['input_node_features'],
    unknown_node_dim=config['output_node_features'],
    out_width=config['out_width'],
).to(device)
nofe_model.load_state_dict(torch.load(
    project_root / 'ERA5' / 'notebooks' / 'subgraph_tkernel_base2_d2_2026-04-25_t13-59-37.pth',
    map_location=device,
))
nofe_model.eval()
print('NOFE model loaded.')

NOFE model loaded.


In [3]:
dates = pd.date_range("2019-01-01", "2019-01-31").strftime("%Y-%m-%d").tolist()

N_in = 200
N_out = 1000

bbox = lon_min, lat_min, lon_max, lat_max

Generator = Data_Generator(project_root / 'ERA5' / 'data')

print('generate data')

graphs = []
for date in dates:
    Generator.load_data(bbox=bbox, vars=vars, filename=f'{date}.nc')
    in_graph = Generator.sample_graph(N=N_in, k=k_in, perplexity=perplexity, sampling_method=input_sampling_method, gen_feature_mode=feature_mode)
    out_graph = Generator.sample_graph(N=N_out, k=k_out, perplexity=perplexity, sampling_method=output_sampling_method, gen_feature_mode=feature_mode)
    cross_graph = Generator.connect_graphs(in_graph=in_graph, out_graph=out_graph, k_cross=k_cross)
    cross_graph["x"] = Generator.construct_output_node_features(cross_graph=cross_graph, interpolate=interpolate_graph_features, gen_feature_mode=feature_mode)
    graphs.append(cross_graph.to(device))

print(f'built {len(graphs)} graphs')

generate data


built 31 graphs


In [4]:
def linear_interp_embedding(emb_source, coords_in, coords_out):
    interp = LinearNDInterpolator(coords_in, emb_source)
    z_out = interp(coords_out)
    mask = np.isnan(z_out)
    z_out[mask] = 0
    return z_out


def idw_interp_embedding(emb_source, coords_in_t, coords_out_t, cross_edge_index):
    # Exactly NOFE's own scheme (ERA5_Graph_Generator.inverse_distance_interpolation),
    # just applied to a precomputed low-dim embedding instead of the raw input features.
    emb_source_t = torch.as_tensor(emb_source, dtype=torch.float32, device=coords_in_t.device)
    z = Generator.inverse_distance_interpolation(
        normed_values_source=emb_source_t,
        normed_coords_source=coords_in_t,
        normed_coords_target=coords_out_t,
        cross_edge_index=cross_edge_index,
    )
    return z.detach().cpu().numpy()


def rbf_interp_embedding(emb_source, coords_in, coords_out):
    rbf = RBFInterpolator(coords_in, emb_source, kernel='thin_plate_spline')
    return rbf(coords_out)


def gp_interp_embedding(emb_source, coords_in, coords_out):
    # Ordinary-kriging stand-in: one independent GP per output dimension, RBF covariance
    # + small nugget (alpha) for numerical stability. length_scale_bounds is kept fairly
    # wide since normed_coords live on a small ~[0, 0.5] scale.
    z_out = np.zeros((coords_out.shape[0], emb_source.shape[1]))
    for d in range(emb_source.shape[1]):
        kernel = RBFKernel(length_scale=0.1, length_scale_bounds=(1e-3, 10.0))
        gp = GaussianProcessRegressor(kernel=kernel, alpha=1e-6, normalize_y=True, random_state=42)
        gp.fit(coords_in, emb_source[:, d])
        z_out[:, d] = gp.predict(coords_out)
    return np.nan_to_num(z_out, nan=0.0, posinf=0.0, neginf=0.0)


INTERP_SCHEMES = ['linear', 'idw', 'rbf', 'gp']

def interp_all_schemes(emb_source, coords_in, coords_out, coords_in_t, coords_out_t, cross_edge_index):
    return {
        'linear': linear_interp_embedding(emb_source, coords_in, coords_out),
        'idw': idw_interp_embedding(emb_source, coords_in_t, coords_out_t, cross_edge_index),
        'rbf': rbf_interp_embedding(emb_source, coords_in, coords_out),
        'gp': gp_interp_embedding(emb_source, coords_in, coords_out),
    }

In [5]:
# NOFE's own construct_output_node_features() always uses IDW to interpolate the *raw* input
# features to query locations before the trained GNN runs. Here we additionally feed the same
# frozen, trained model query features built by linear / RBF / GP interpolation of the raw
# features instead, to see how sensitive NOFE itself is to the interpolation scheme used at its
# input stage (the model was trained only ever seeing IDW-interpolated inputs, so linear/rbf/gp
# here constitute a mild distribution shift, not just "another baseline"). `nofe_idw` is NOFE
# exactly as normally run.
pred_nofe = {s: [] for s in INTERP_SCHEMES}

with torch.no_grad():
    for i, sample in enumerate(graphs):
        X = sample.source.normed_values.detach().cpu().numpy()
        coords_in_t = sample.source.normed_coords.detach()
        coords_out_t = sample.target.normed_coords.detach()
        coords_in = coords_in_t.cpu().numpy()
        coords_out = coords_out_t.cpu().numpy()
        cross_edge_index = sample.cross.edge_index

        original_target_x = sample.target.x.clone()
        for scheme, X_query in interp_all_schemes(X, coords_in, coords_out, coords_in_t, coords_out_t, cross_edge_index).items():
            sample.target.x = torch.as_tensor(X_query, dtype=torch.float32, device=device)
            pred_nofe[scheme].append(nofe_model(sample).detach().cpu().numpy())
        sample.target.x = original_target_x  # restore NOFE's own IDW-based input for downstream cells

print(f'computed NOFE predictions under {list(pred_nofe.keys())} input-interpolation schemes for {len(graphs)} graphs')

computed NOFE predictions under ['linear', 'idw', 'rbf', 'gp'] input-interpolation schemes for 31 graphs


In [6]:
pred_pca = {s: [] for s in INTERP_SCHEMES}
pred_tsne = {s: [] for s in INTERP_SCHEMES}
pred_umap = {s: [] for s in INTERP_SCHEMES}
pred_umap['native'] = []

pca_idw_native_max_diff = []  # sanity check: PCA(IDW(raw)) vs IDW(PCA(raw))

for i, sample in enumerate(graphs):
    print(f'{i}/{len(graphs)}')

    X = sample.source.normed_values.detach().cpu().numpy()
    coords_in_t = sample.source.normed_coords.detach()
    coords_out_t = sample.target.normed_coords.detach()
    coords_in = coords_in_t.cpu().numpy()
    coords_out = coords_out_t.cpu().numpy()
    cross_edge_index = sample.cross.edge_index

    # Raw-feature IDW interpolation to query locations -- mirrors NOFE's own query-feature
    # initialization exactly (same neighbours, same weights).
    X_query_idw = Generator.inverse_distance_interpolation(
        normed_values_source=sample.source.normed_values,
        normed_coords_source=coords_in_t,
        normed_coords_target=coords_out_t,
        cross_edge_index=cross_edge_index,
    ).detach().cpu().numpy()

    # PCA
    print('PCA ...', end='\r')
    pca = PCA(n_components=3)
    pca_source = pca.fit_transform(X)
    for scheme, z in interp_all_schemes(pca_source, coords_in, coords_out, coords_in_t, coords_out_t, cross_edge_index).items():
        pred_pca[scheme].append(z)
    pca_idw_native_max_diff.append(np.abs(pca.transform(X_query_idw) - pred_pca['idw'][-1]).max())

    # t-SNE (no native out-of-sample extension -- only linear/idw/rbf/gp on the embedding)
    print('t-SNE ...', end='\r')
    tsne_source = TSNE(n_components=3, perplexity=30, learning_rate="auto", random_state=42).fit_transform(X)
    for scheme, z in interp_all_schemes(tsne_source, coords_in, coords_out, coords_in_t, coords_out_t, cross_edge_index).items():
        pred_tsne[scheme].append(z)

    # UMAP -- umap-learn is installed directly in this env, run in-process
    print('UMAP ...', end='\r')
    reducer = umap.UMAP(n_components=3, n_neighbors=15, min_dist=0.1, metric='euclidean', random_state=42)
    umap_source = reducer.fit_transform(X)
    for scheme, z in interp_all_schemes(umap_source, coords_in, coords_out, coords_in_t, coords_out_t, cross_edge_index).items():
        pred_umap[scheme].append(z)
    pred_umap['native'].append(reducer.transform(X_query_idw))

print(f'\nmax |PCA(IDW(raw)) - IDW(PCA(raw))| across all dates: {max(pca_idw_native_max_diff):.2e} (should be ~0, confirms exact equivalence for the linear PCA baseline)')


max |PCA(IDW(raw)) - IDW(PCA(raw))| across all dates: 9.54e-07 (should be ~0, confirms exact equivalence for the linear PCA baseline)


## Extending the same 4 interpolation schemes to PHATE / spatialPCA / Concord / fPCA

These four methods aren't named in reviewer's question, but they were added as extra baselines
in `rebuttal_distance_presevation_super_resolution.ipynb`, which still only carries their
embeddings to query locations via `LinearNDInterpolator`. Since the interpolation-scheme choice
turned out to change the PCA/t-SNE/UMAP numbers by an order of magnitude, leaving these four on the
old (`linear`-only) scheme would make the overall comparison inconsistent -- so they get the same
`linear` / `idw` / `rbf` / `gp` treatment here for a single, internally consistent table.

In [7]:
# Start the spatialPCA R process ONCE, outside the loop -- library(SpatialPCA) (Seurat, SPARK,
# etc.) is the slow part to load. It stays alive and answers one request per line over
# stdin/stdout for the whole loop below (same server pattern as in
# rebuttal_distance_presevation_super_resolution.ipynb).
spdir = f'{project_root}/ERA5/spatialpca'
spatialpca_call_id = f'spatialpca_buffer_{os.getpid()}'
spatialpca_proc = subprocess.Popen(
    ['conda', 'run', '-n', 'spatialpca', '--no-capture-output', 'Rscript', 'execute_spatialpca_server.R'],
    cwd=spdir, stdin=subprocess.PIPE, stdout=subprocess.PIPE, stderr=subprocess.PIPE,
    text=True, bufsize=1,
)
ready = spatialpca_proc.stdout.readline().strip()
if ready != 'READY':
    raise RuntimeError(f'spatialPCA server failed to start: {ready!r}\n{spatialpca_proc.stderr.read()}')
print('spatialPCA server ready.')


def spatialpca_embed(X, coords, call_id, n_components=3):
    np.savetxt(f'{spdir}/{call_id}_X.csv', X, delimiter=',')
    np.savetxt(f'{spdir}/{call_id}_coords.csv', coords, delimiter=',')

    spatialpca_proc.stdin.write(f'{call_id} {n_components}\n')
    spatialpca_proc.stdin.flush()
    response = spatialpca_proc.stdout.readline().strip()

    os.remove(f'{spdir}/{call_id}_X.csv')
    os.remove(f'{spdir}/{call_id}_coords.csv')

    if response != 'OK':
        raise RuntimeError(f'spatialPCA failed for {call_id}: {response}')

    Y = np.loadtxt(f'{spdir}/res_{call_id}.csv', delimiter=',', skiprows=1)
    os.remove(f'{spdir}/res_{call_id}.csv')
    return Y


def concord_embed(X, n_components=3):
    adata = ad.AnnData(X=X.astype(np.float32))
    cur_ccd = ccd.Concord(
        adata=adata, input_feature=list(adata.var_names), device=device,
        latent_dim=n_components, verbose=False,
        # preload_dense=True forces num_workers to 0 in concord-sc's DataLoaderManager --
        # without it, every worker re-imports torch/anndata/concord from scratch per call.
        preload_dense=True,
    )
    cur_ccd.fit_transform(output_key='Concord')
    return adata.obsm['Concord']


def fpca_embed(X, n_components=3):
    generator = syn.FPCADataGenerator()
    generator.fit(X.astype(np.float64), n_fpca_components=n_components)
    return generator.eig_scores


pred_phate = {s: [] for s in INTERP_SCHEMES}
pred_spatialpca = {s: [] for s in INTERP_SCHEMES}
pred_concord = {s: [] for s in INTERP_SCHEMES}
pred_fpca = {s: [] for s in INTERP_SCHEMES}

for i, sample in enumerate(graphs):
    print(f'{i}/{len(graphs)}')

    X = sample.source.normed_values.detach().cpu().numpy()
    coords_in_t = sample.source.normed_coords.detach()
    coords_out_t = sample.target.normed_coords.detach()
    coords_in = coords_in_t.cpu().numpy()
    coords_out = coords_out_t.cpu().numpy()
    cross_edge_index = sample.cross.edge_index

    # PHATE
    print('PHATE ...', end='\r')
    phate_source = phate.PHATE(
        n_components=3, knn=15, mds_solver='smacof', random_state=42, verbose=False
    ).fit_transform(X)
    for scheme, z in interp_all_schemes(phate_source, coords_in, coords_out, coords_in_t, coords_out_t, cross_edge_index).items():
        pred_phate[scheme].append(z)

    # spatialPCA
    print('spatialPCA ...', end='\r')
    spatialpca_source = spatialpca_embed(X, coords_in, spatialpca_call_id, n_components=3)
    for scheme, z in interp_all_schemes(spatialpca_source, coords_in, coords_out, coords_in_t, coords_out_t, cross_edge_index).items():
        pred_spatialpca[scheme].append(z)

    # Concord
    print('Concord ...', end='\r')
    concord_source = concord_embed(X, n_components=3)
    for scheme, z in interp_all_schemes(concord_source, coords_in, coords_out, coords_in_t, coords_out_t, cross_edge_index).items():
        pred_concord[scheme].append(z)

    # functionalPCA
    print('fPCA ...', end='\r')
    fpca_source = fpca_embed(X, n_components=3)
    for scheme, z in interp_all_schemes(fpca_source, coords_in, coords_out, coords_in_t, coords_out_t, cross_edge_index).items():
        pred_fpca[scheme].append(z)

spatialpca_proc.stdin.write('QUIT\n')
spatialpca_proc.stdin.flush()
spatialpca_proc.wait(timeout=10)
print('\ndone')


done


In [8]:
def pairwise_distances(X):
    norms = np.sum(X**2, axis=1, keepdims=True)
    dist2 = norms + norms.T - 2 * X @ X.T
    dist2 = np.maximum(dist2, 0.0)
    return np.sqrt(dist2)

def distance_preservation(H, L):
    DH = pairwise_distances(H)
    DL = pairwise_distances(L)
    i = np.triu_indices_from(DH, k=1)
    return np.corrcoef(DH[i], DL[i])[0, 1]

def stress(H, L):
    DH = pairwise_distances(H)
    DL = pairwise_distances(L)
    i = np.triu_indices_from(DH, k=1)
    dH, dL = DH[i], DL[i]
    return np.sqrt(np.sum((dH - dL)**2) / np.sum(dH**2))

def edge_distances(X, edge_indices):
    # edge_indices is PyG's [2, E] layout (row 0 = source nodes, row 1 = target nodes).
    diff = X[edge_indices[0]] - X[edge_indices[1]]
    return np.linalg.norm(diff, axis=1)

def edge_distance_preservation(H, L, edge_indices):
    dH = edge_distances(H, edge_indices)
    dL = edge_distances(L, edge_indices)
    return np.corrcoef(dH, dL)[0, 1]

def edge_stress(H, L, edge_indices):
    dH = edge_distances(H, edge_indices)
    dL = edge_distances(L, edge_indices)
    return np.sqrt(np.sum((dH - dL)**2) / np.sum(dH**2))

In [9]:
def mean_std(arr):
    return f"{np.mean(arr):.4f} ± {np.std(arr):.4f}"


def compute_scores(pred_dict, method_name):
    rows = {}
    for scheme, preds in pred_dict.items():
        preservation, strs, edge_preservation, edge_strs = [], [], [], []
        for i in range(len(graphs)):
            edge_indices = graphs[i].target.edge_index.detach().cpu().numpy()
            H = graphs[i].target.meta.x_true.detach().cpu().numpy()
            L = preds[i]
            preservation.append(distance_preservation(H, L))
            strs.append(stress(H, L))
            edge_preservation.append(edge_distance_preservation(H, L, edge_indices))
            edge_strs.append(edge_stress(H, L, edge_indices))
        rows[f'{method_name}_{scheme}'] = [
            mean_std(preservation), mean_std(strs), mean_std(edge_preservation), mean_std(edge_strs),
        ]
    return rows


all_scores = {}
all_scores.update(compute_scores(pred_nofe, 'nofe'))
all_scores.update(compute_scores(pred_pca, 'pca'))
all_scores.update(compute_scores(pred_tsne, 'tsne'))
all_scores.update(compute_scores(pred_umap, 'umap'))
all_scores.update(compute_scores(pred_phate, 'phate'))
all_scores.update(compute_scores(pred_spatialpca, 'spatialpca'))
all_scores.update(compute_scores(pred_concord, 'concord'))
all_scores.update(compute_scores(pred_fpca, 'fpca'))

scores = pd.DataFrame(
    all_scores,
    index=["corr (global)", "stress (global)", "corr (local)", "stress (local)"],
).T

pd.set_option('display.width', 120)
pd.set_option('display.max_rows', 100)
print(scores)

/var/folders/px/747r5b694x947qk1vdh9b3th0000gp/T/ipykernel_18475/693355063.py:3: RuntimeWarning: divide by zero encountered in matmul
  dist2 = norms + norms.T - 2 * X @ X.T
/var/folders/px/747r5b694x947qk1vdh9b3th0000gp/T/ipykernel_18475/693355063.py:3: RuntimeWarning: overflow encountered in matmul
  dist2 = norms + norms.T - 2 * X @ X.T
/var/folders/px/747r5b694x947qk1vdh9b3th0000gp/T/ipykernel_18475/693355063.py:3: RuntimeWarning: invalid value encountered in matmul
  dist2 = norms + norms.T - 2 * X @ X.T


                     corr (global)  stress (global)     corr (local)   stress (local)
nofe_linear        0.3348 ± 0.0671  0.6667 ± 0.0221  0.5032 ± 0.0438  0.7098 ± 0.0192
nofe_idw           0.4958 ± 0.0705  0.4816 ± 0.0291  0.5394 ± 0.0439  0.6576 ± 0.0219
nofe_rbf           0.4714 ± 0.0629  0.4309 ± 0.0229  0.5354 ± 0.0436  0.6052 ± 0.0219
nofe_gp            0.4567 ± 0.0609  0.5038 ± 0.0263  0.5318 ± 0.0401  0.6271 ± 0.0208
pca_linear         0.0144 ± 0.0349  0.9948 ± 0.0024  0.0108 ± 0.0436  0.9961 ± 0.0036
pca_idw            0.6671 ± 0.0443  0.4068 ± 0.0219  0.3840 ± 0.0508  0.8175 ± 0.0157
pca_rbf            0.6728 ± 0.0498  0.3612 ± 0.0225  0.4931 ± 0.0509  0.7372 ± 0.0221
pca_gp             0.5587 ± 0.0628  0.4911 ± 0.0437  0.4346 ± 0.0574  0.7230 ± 0.0234
tsne_linear        0.0071 ± 0.0283  1.1803 ± 0.2982  0.0058 ± 0.0412  1.5533 ± 0.8672
tsne_idw           0.5288 ± 0.0640  3.9712 ± 5.0097  0.3277 ± 0.0517  1.7143 ± 1.5335
tsne_rbf           0.5413 ± 0.0572  4.4199 ± 5.5665  0

## "Interpolate raw features first, then embed" -- matching NOFE's actual pipeline order

Everything above (the `_linear/_idw/_rbf/_gp` columns for PCA/t-SNE/UMAP/PHATE/spatialPCA/Concord/fPCA)
embeds the source points first and only afterwards interpolates that embedding to query locations --
this is what the reviewer's wording literally asks for ("baseline embeddings followed by the same
interpolation scheme"), and for PCA it is provably identical to interpolating first (proven above).

But NOFE's own pipeline runs in the *opposite* order: interpolate the raw features to query
locations first, then embed. For nonlinear/non-parametric methods (t-SNE, UMAP, PHATE, spatialPCA,
Concord), these two orderings are **not** equivalent -- `embed(interpolate(raw))` need not equal
`interpolate(embed(raw))` unless the embedding is affine, which is a PCA-only property.

Below we add the missing, structurally-matched comparison: for each interpolation scheme, we
interpolate the raw input features to query locations (exactly as already done, reusing
`interp_all_schemes` on the raw features), append the resulting synthetic query points to the
source point set, and **jointly re-fit** each embedding method on the combined (source + query)
data -- then read off the embedding rows assigned to the query points. This is the closest
faithful reconstruction of "interpolate first, then embed" for methods (t-SNE, PHATE, spatialPCA,
Concord) that have no `.transform()` for new points; UMAP gets both this joint-refit version
(`umap_pre_*`) and its native `.transform()`-based version (`umap_native`, computed above) for
comparison, since these are two different ways of embedding after interpolating.

These are new columns, added alongside (not replacing) the `embed-then-interpolate` columns above.

**`linear` is excluded from this pass.** Checked directly: with only 200 source points scattered
across the whole bounding box, **~99% of the 1000 query points fall outside the convex hull** of
the source points in `normed_coords` space. `LinearNDInterpolator` returns NaN for every point
outside the hull, which `linear_interp_embedding`'s mask silently zeros out -- so for the `linear`
scheme, ~990 of the 1000 "interpolated" raw-feature rows are the exact same all-zero vector. Stacked
into the joint (source + query) dataset, that's not "a worse interpolation scheme," it's ~990
duplicate rows, which breaks density/kernel-based fitting outright: PHATE warns of tens of thousands
of zero-distance point pairs, and spatialPCA's automatic (Sheather-Jones) bandwidth selection throws
`assignment of NULL is not valid for @'bandwidth'` and crashes. This 99%-outside-the-hull number is
also, in hindsight, the sharper explanation for why every `*_linear` column throughout this whole
notebook (and the original submission's baseline table) is close to zero -- it isn't merely "a less
accurate interpolation," it's "no interpolation at all" for nearly every query point.

In [10]:
def tsne_pre_embed(X_source, X_query, n_components=3):
    X_combined = np.vstack([X_source, X_query])
    emb = TSNE(n_components=n_components, perplexity=30, learning_rate="auto", random_state=42).fit_transform(X_combined)
    return emb[X_source.shape[0]:]


def umap_pre_embed(X_source, X_query, n_components=3):
    X_combined = np.vstack([X_source, X_query])
    reducer = umap.UMAP(n_components=n_components, n_neighbors=15, min_dist=0.1, metric='euclidean', random_state=42)
    emb = reducer.fit_transform(X_combined)
    return emb[X_source.shape[0]:]


def phate_pre_embed(X_source, X_query, n_components=3):
    X_combined = np.vstack([X_source, X_query])
    emb = phate.PHATE(n_components=n_components, knn=15, mds_solver='smacof', random_state=42, verbose=False).fit_transform(X_combined)
    return emb[X_source.shape[0]:]


def spatialpca_pre_embed(X_source, X_query, coords_source, coords_query, call_id, n_components=3):
    X_combined = np.vstack([X_source, X_query])
    coords_combined = np.vstack([coords_source, coords_query])
    emb = spatialpca_embed(X_combined, coords_combined, call_id, n_components=n_components)
    return emb[X_source.shape[0]:]


def concord_pre_embed(X_source, X_query, n_components=3):
    X_combined = np.vstack([X_source, X_query])
    emb = concord_embed(X_combined, n_components=n_components)
    return emb[X_source.shape[0]:]


# 'linear' is excluded here: ~99% of query points fall outside the convex hull of the 200
# source points in this normed_coords space, so LinearNDInterpolator returns NaN (silently
# zeroed by linear_interp_embedding's mask) for nearly every query point. Stacking ~990
# identical all-zero rows into the joint (source + query) dataset isn't "a worse
# interpolation scheme" -- it's degenerate duplicate data that breaks density/kernel-based
# fitting outright (confirmed: PHATE warns of tens of thousands of zero-distance pairs;
# spatialPCA's automatic (Sheather-Jones) bandwidth selection throws
# "assignment of NULL is not valid for @'bandwidth'" and crashes).
PRE_INTERP_SCHEMES = ['idw', 'rbf', 'gp']

pred_tsne_pre = {s: [] for s in PRE_INTERP_SCHEMES}
pred_umap_pre = {s: [] for s in PRE_INTERP_SCHEMES}
pred_phate_pre = {s: [] for s in PRE_INTERP_SCHEMES}
pred_spatialpca_pre = {s: [] for s in PRE_INTERP_SCHEMES}
pred_concord_pre = {s: [] for s in PRE_INTERP_SCHEMES}

# Fresh spatialPCA server for this pass -- the one started earlier already received QUIT.
spatialpca_pre_call_id = f'spatialpca_pre_buffer_{os.getpid()}'
spatialpca_proc = subprocess.Popen(
    ['conda', 'run', '-n', 'spatialpca', '--no-capture-output', 'Rscript', 'execute_spatialpca_server.R'],
    cwd=spdir, stdin=subprocess.PIPE, stdout=subprocess.PIPE, stderr=subprocess.PIPE,
    text=True, bufsize=1,
)
ready = spatialpca_proc.stdout.readline().strip()
if ready != 'READY':
    raise RuntimeError(f'spatialPCA server failed to start: {ready!r}\n{spatialpca_proc.stderr.read()}')
print('spatialPCA server ready (pre-embedding pass).')

for i, sample in enumerate(graphs):
    print(f'{i}/{len(graphs)}')

    X = sample.source.normed_values.detach().cpu().numpy()
    coords_in_t = sample.source.normed_coords.detach()
    coords_out_t = sample.target.normed_coords.detach()
    coords_in = coords_in_t.cpu().numpy()
    coords_out = coords_out_t.cpu().numpy()
    cross_edge_index = sample.cross.edge_index

    all_schemes = interp_all_schemes(X, coords_in, coords_out, coords_in_t, coords_out_t, cross_edge_index)
    for scheme in PRE_INTERP_SCHEMES:
        X_query = all_schemes[scheme]

        print(f'{scheme}: t-SNE ...', end='\r')
        pred_tsne_pre[scheme].append(tsne_pre_embed(X, X_query))

        print(f'{scheme}: UMAP ...', end='\r')
        pred_umap_pre[scheme].append(umap_pre_embed(X, X_query))

        print(f'{scheme}: PHATE ...', end='\r')
        pred_phate_pre[scheme].append(phate_pre_embed(X, X_query))

        print(f'{scheme}: spatialPCA ...', end='\r')
        pred_spatialpca_pre[scheme].append(spatialpca_pre_embed(X, X_query, coords_in, coords_out, spatialpca_pre_call_id))

        print(f'{scheme}: Concord ...', end='\r')
        pred_concord_pre[scheme].append(concord_pre_embed(X, X_query))

spatialpca_proc.stdin.write('QUIT\n')
spatialpca_proc.stdin.flush()
spatialpca_proc.wait(timeout=10)
print('\ndone (pre-embedding, joint-refit pass)')


done (pre-embedding, joint-refit pass)


In [11]:
# New table only -- does not touch or recompute `all_scores` / `scores` from above.
pre_scores_dict = {}
pre_scores_dict.update(compute_scores(pred_tsne_pre, 'tsne_pre'))
pre_scores_dict.update(compute_scores(pred_umap_pre, 'umap_pre'))
pre_scores_dict.update(compute_scores(pred_phate_pre, 'phate_pre'))
pre_scores_dict.update(compute_scores(pred_spatialpca_pre, 'spatialpca_pre'))
pre_scores_dict.update(compute_scores(pred_concord_pre, 'concord_pre'))

pre_scores = pd.DataFrame(
    pre_scores_dict,
    index=["corr (global)", "stress (global)", "corr (local)", "stress (local)"],
).T

print(pre_scores)

/var/folders/px/747r5b694x947qk1vdh9b3th0000gp/T/ipykernel_18475/693355063.py:3: RuntimeWarning: divide by zero encountered in matmul
  dist2 = norms + norms.T - 2 * X @ X.T
/var/folders/px/747r5b694x947qk1vdh9b3th0000gp/T/ipykernel_18475/693355063.py:3: RuntimeWarning: overflow encountered in matmul
  dist2 = norms + norms.T - 2 * X @ X.T
/var/folders/px/747r5b694x947qk1vdh9b3th0000gp/T/ipykernel_18475/693355063.py:3: RuntimeWarning: invalid value encountered in matmul
  dist2 = norms + norms.T - 2 * X @ X.T


                      corr (global)  stress (global)     corr (local)   stress (local)
tsne_pre_idw        0.4697 ± 0.0595  2.5869 ± 0.1844  0.2027 ± 0.0443  0.9143 ± 0.0909
tsne_pre_rbf        0.5047 ± 0.0531  2.6408 ± 0.2085  0.2841 ± 0.0460  1.0082 ± 0.1322
tsne_pre_gp         0.5138 ± 0.0447  2.5356 ± 0.2147  0.3040 ± 0.0504  1.0433 ± 0.1198
umap_pre_idw        0.4045 ± 0.0672  1.2278 ± 0.1415  0.1369 ± 0.0432  0.9261 ± 0.0403
umap_pre_rbf        0.4564 ± 0.0512  0.8675 ± 0.1279  0.2459 ± 0.0362  0.8547 ± 0.0188
umap_pre_gp         0.4696 ± 0.0628  0.7796 ± 0.1178  0.2505 ± 0.0387  0.8317 ± 0.0238
phate_pre_idw       0.4626 ± 0.0731  0.9920 ± 0.0002  0.1920 ± 0.0454  0.9985 ± 0.0002
phate_pre_rbf       0.4854 ± 0.0666  0.9921 ± 0.0002  0.3017 ± 0.0400  0.9984 ± 0.0002
phate_pre_gp        0.4904 ± 0.0671  0.9921 ± 0.0002  0.3022 ± 0.0430  0.9981 ± 0.0002
spatialpca_pre_idw  0.3272 ± 0.0584  0.7536 ± 0.2144  0.3312 ± 0.0571  0.9385 ± 0.2103
spatialpca_pre_rbf  0.2565 ± 0.0580  1.2847

## Notes

- `nofe_*`: the real trained model (`subgraph_tkernel_base2_d2_2026-04-25_t13-59-37.pth`). NOFE
  always interpolates raw features first, then embeds -- there's no other way to run it, so the
  4 variants (`linear/idw/rbf/gp`) test its sensitivity to that choice. `nofe_idw` is NOFE as
  normally run.
- `pca_*` / `fpca_*`: for PCA, interpolating before or after the embedding gives identical results
  (affine transform, proven and verified numerically), so these columns already cover both
  orderings. fPCA turned out to be numerically identical to PCA (a library quirk on non-functional
  input), so it's redundant with `pca_*`.
- `tsne_*` / `umap_*` / `phate_*` / `spatialpca_*` / `concord_*`: embed the source points first,
  then interpolate that embedding to query locations. `umap_native` uses UMAP's own `.transform()`
  instead of interpolating.
- `*_pre_*` (separate table below): the opposite ordering for those same 5 nonlinear methods --
  interpolate raw features first, then jointly re-fit the embedding on source+query points
  together (the ordering that structurally matches NOFE). `linear` is excluded from this pass:
  ~99% of query points fall outside the 200 source points' convex hull, so `LinearNDInterpolator`
  returns all-zero duplicates that crash spatialPCA's fitting.

## Summary of results

- Interpolation scheme matters a lot: `linear` gives near-zero correlation (0.01-0.04) for every
  method; `idw`/`rbf`/`gp` raise it to 0.2-0.67.
- **Global** structure preservation: PCA (`idw`/`rbf`) is best (corr ≈ 0.67), ahead of NOFE
  (corr ≈ 0.47-0.50) and every other baseline, under every scheme and ordering tested.
- **Local** structure preservation: NOFE is best (corr ≈ 0.54, stress ≈ 0.66), ahead of every
  baseline, under every scheme and ordering tested.
- NOFE is sensitive to a bad input-interpolation choice (triangulation hurts it) but robust to a
  better one (RBF matches or improves on its native IDW, despite never being trained on it).
- **Conclusion**: NOFE's advantage is specific to local structure, not global -- where a
  properly-interpolated PCA is the strongest method overall.